# 🔊 Volume Analysis

발표 중 **음량이 적절한지 판단하기 위한 분석 모듈**

#### 주요 입력값
- `noise_db`: 사전 테스트에서 사람이 말하지 않았을 때의 주변 소음
- `baseline_voice_db`: 캘리브레이션에서 사용자가 평소 목소리로 말했을 때의 음량
- `current_db`: 발표 중 현재 측정된 음량

#### 주요 계산값
- `relative_db = current_db - noise_db`
- `baseline_difference_db = current_db - baseline_voice_db`

> 현재는 threshold와 최종 판단 로직을 확정하지 않고, 여러 지표를 확인하면서 적절한 평가 방법을 찾는 것을 목표로 함.

In [8]:
from pydantic import BaseModel

In [9]:
class Calibration(BaseModel):
    noise_db: float
    baseline_voice_db: float

class VolumeMeasurement(BaseModel):
    current_db: float
    relative_db: float
    baseline_difference_db: float

#### Calibration

In [10]:
calibration = Calibration(
    noise_db=45.0,
    baseline_voice_db=65.0,
)

print('주변 소음:', calibration.noise_db, 'dB')
print('기준 목소리:', calibration.baseline_voice_db, 'dB')
print('기준 목소리 - 주변 소음:', calibration.baseline_voice_db - calibration.noise_db, 'dB')

주변 소음: 45.0 dB
기준 목소리: 65.0 dB
기준 목소리 - 주변 소음: 20.0 dB


#### 실시간 측정

In [11]:
def measure_volume(current_db: float, calibration: Calibration):
    relative_db = current_db - calibration.noise_db
    baseline_difference_db = current_db - calibration.baseline_voice_db

    return VolumeMeasurement(
        current_db=current_db,
        relative_db=relative_db,
        baseline_difference_db=baseline_difference_db,
    )

measurement = measure_volume(current_db=58.0, calibration=calibration)
measurement

VolumeMeasurement(current_db=58.0, relative_db=13.0, baseline_difference_db=-7.0)

#### 절대 음량 판단

In [12]:
# 논문 및 테스트 데이터를 바탕으로 수정
ABSOLUTE_LOW = 60
ABSOLUTE_HIGH = 75

def judge_absolute_volume(current_db: float) -> str:
    if current_db < ABSOLUTE_LOW:
        return 'low'
    elif current_db > ABSOLUTE_HIGH:
        return 'high'
    return 'normal'

judge_absolute_volume(measurement.current_db)

'low'

#### 상대 음량 판단 (주변 소음 대비)

In [13]:
# 실제 기준 조사 후 수정
RELATIVE_LOW = 10

def judge_relative_volume(relative_db: float) -> str:
    if relative_db < RELATIVE_LOW:
        return 'low'
    return 'normal'

judge_relative_volume(measurement.relative_db)

'normal'

#### 캘리브레이션 대비 변화
- 캘리브레이션의 상대 음량 - 현재 상대 음량
- 아직 미사용

In [14]:
def calculate_volume_features(current_db: float, calibration: Calibration):
    return {
        'current_db': current_db,
        'relative_db': current_db - calibration.noise_db,
        'baseline_difference_db': current_db - calibration.baseline_voice_db,
    }

#### 테스트

In [15]:
test_values = [50, 55, 60, 65, 70, 75, 80]

for current_db in test_values:
    features = calculate_volume_features(current_db, calibration)
    print(
        f"current={features['current_db']:>5.1f} dB | "
        f"relative={features['relative_db']:>5.1f} dB | "
        f"baseline_diff={features['baseline_difference_db']:>5.1f} dB"
    )

current= 50.0 dB | relative=  5.0 dB | baseline_diff=-15.0 dB
current= 55.0 dB | relative= 10.0 dB | baseline_diff=-10.0 dB
current= 60.0 dB | relative= 15.0 dB | baseline_diff= -5.0 dB
current= 65.0 dB | relative= 20.0 dB | baseline_diff=  0.0 dB
current= 70.0 dB | relative= 25.0 dB | baseline_diff=  5.0 dB
current= 75.0 dB | relative= 30.0 dB | baseline_diff= 10.0 dB
current= 80.0 dB | relative= 35.0 dB | baseline_diff= 15.0 dB


-- FE 측과 dB 수치에 대해 더 논의 후 추후 개발 예정